# TI YOLOv9-S-Lite: flame detection on Colab
Select **Runtime → Change runtime type → GPU**, then run the cells in order.
Google Drive authorization is interactive. Use the account selected for this project.

This notebook embeds the experiment source and six checksum-pinned TI modules.
No GitHub push, manual labeling, or local dataset upload is required. It can reuse
the teammate's Drive dataset or download the public Kaggle dataset.

Three short learning-rate pilots use training data and monitor validation only.
The best pilot LR starts a fresh full run (up to 40 epochs, EMA, cosine schedule,
mixed precision, checkpoint recovery). This is a bounded search, not a claim of
globally optimal hyperparameters. GPU memory determines microbatch size while
effective batch stays 32. A full-resolution forward/backward/export preflight
must pass before pilots. The test split is not evaluated by Run all.

In [ ]:
import base64, hashlib, io, json, pathlib, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '495f2025f43ee1b1c81a906fc1095ea550c08c3f1b007b472267417cad47d357'
ROOT = pathlib.Path('/content/honeywell_yolov9')
ROOT.mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(payload)).extractall(ROOT)
WF = ROOT/'workflow'
TI = ROOT/'tensorlab'
RUN = pathlib.Path('/content/drive/MyDrive/HoneywellTraining/experiments/yolov9_s_ti_lite_v1')
RUN.mkdir(parents=True, exist_ok=True)
(RUN/'source_bundle.zip').write_bytes(payload)
print('Experiment:', RUN, '\nSource bundle SHA256:', '495f2025f43ee1b1c81a906fc1095ea550c08c3f1b007b472267417cad47d357')

**Install the isolated, pinned training environment.** This can take several minutes.
The Colab notebook kernel stays separate; training uses the environment below.

In [ ]:
ENV = pathlib.Path('/content/yolov9_env_py311')
# Colab's system Python may omit ensurepip; uv creates/seeds the environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.5'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--seed', '--allow-existing',
                '--python', '3.11', str(ENV)], check=True)
PY = str(ENV/'bin/python')
def run(*args):
    command = [PY, *map(str, args)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, command)
run('-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1',
    '--index-url', 'https://download.pytorch.org/whl/cu124')
run('-m', 'pip', 'install', '-r', WF/'requirements.txt', 'kagglehub==0.3.13')
run('-c', "import torch; assert torch.cuda.is_available(), 'Enable a GPU runtime'; print(torch.__version__, torch.cuda.get_device_name())")
subprocess.run(['nvidia-smi'], check=True)

**Obtain the verified pretrained checkpoint and data.** The existing Drive copy is
preferred when present. Otherwise Kaggle is downloaded automatically. ZIP_PATH is
optional if dataset.zip already exists in Drive. Originals are preserved.

In [ ]:
WEIGHTS = ROOT/'yolov9_s_ti_lite_coco.pth'
DATA = pathlib.Path('/content/yolov9_dataset')
MANIFESTS = RUN/'manifests'
DRIVE_DATA = '/content/drive/MyDrive/HoneywellTraining/datasets/fasdd-cv-coco'
ZIP_PATH = ''  # Optional existing Drive ZIP path; leave empty for Drive folder/Kaggle.
run(WF/'bootstrap.py', 'weights', '--destination', WEIGHTS)
command = [WF/'bootstrap.py', 'dataset', '--destination', DATA, '--drive-source', DRIVE_DATA]
if ZIP_PATH:
    command += ['--archive', ZIP_PATH]
run(*command)
run(WF/'data.py', '--root', DATA, '--output', MANIFESTS)
print((MANIFESTS/'prepare_report.json').read_text()[:5000])

**Preflight.** Verify source/checkpoint loading, empty-label negatives, gradients,
metrics, a real 640-pixel training step, and ONNX parity. No full run starts if a
check fails. Any adaptive batch reduction is recorded before training begins.

In [ ]:
run(WF/'test_workflow.py', '--tensorlab', TI, '--checkpoint', WEIGHTS, '--export')
run(WF/'preflight.py', '--tensorlab', TI, '--checkpoint', WEIGHTS,
    '--manifests', MANIFESTS, '--output', RUN/'preflight')
print((RUN/'preflight/preflight.json').read_text())

**Train.** Run three LR pilots, select by monitor-validation COCO AP, then train
the full split from the verified COCO initialization. Rerunning this cell resumes
matching checkpoints. The source, recipe, manifests, and checkpoint identity must
match. Keep this cell running; logs/checkpoints are saved to Drive.

In [ ]:
CONFIG = RUN/'preflight/selected_config.json'
LOG = RUN/'training_console.log'
command = [PY, '-u', str(WF/'train.py'), '--config', str(CONFIG),
           '--tensorlab', str(TI), '--checkpoint', str(WEIGHTS),
           '--manifests', str(MANIFESTS), '--output', str(RUN/'training'), '--tune']
with LOG.open('a', buffering=1) as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    status = process.wait()
if status:
    raise RuntimeError(f'Training stopped (exit {status}). See {LOG}; rerun to resume.')

**Full validation and ONNX handover.** Produces measured COCO AP50:95, AP50,
AP75, size-specific AP/AR, and precision/recall/F1 with TP/FP/FN at confidence
0.40 and matching IoU 0.50. Missing size groups are null, not invented scores.
Board performance and 1 TOPS compliance remain separate measurements.

In [ ]:
run(WF/'finish.py', '--tensorlab', TI, '--checkpoint', RUN/'training/full/best.pth',
    '--manifests', MANIFESTS, '--output', RUN/'validation', '--split', 'validation_full')
print((RUN/'validation/REPORT.md').read_text())
print('ONNX:', RUN/'validation/export/flame.onnx')
print('Contract:', RUN/'validation/export/model_contract.json')

**Optional fixed-model test evaluation is intentionally separate.** The Kaggle
test has already been examined by the team; it is not a new untouched benchmark.
After model/recipe/threshold decisions are frozen, use finish.py with
--split test --final-test --skip-export in a separate cell. Do not tune on it.